
# 23 — Preference Data, Sequence Log-Probabilities, and Reward Modeling

## Goal

Understand how language-model post-training can learn from
preferences between complete responses.

We will build preference data representations, calculate sequence
log-probabilities, derive the Bradley-Terry preference objective,
and train a small reward model.

## 01. Preference Data and Pairwise Comparisons

A preference example contains a prompt, a chosen response,
and a rejected response.

Preference is conditional on the prompt. It does not imply
that one response is universally better than another.


In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PreferenceExample:
    """Represent one prompt-conditioned pairwise preference.

    Args:
        prompt: Instruction shared by both responses.
        chosen: Preferred response.
        rejected: Less preferred response.
    """

    prompt: str
    chosen: str
    rejected: str


preferences = [
    PreferenceExample("Say yes", "Yes", "No"),
    PreferenceExample("Say no", "No", "Yes"),
    PreferenceExample("Say hello", "Hello", "Farewell"),
]
for example in preferences:
    print(example)

PreferenceExample(prompt='Say yes', chosen='Yes', rejected='No')
PreferenceExample(prompt='Say no', chosen='No', rejected='Yes')
PreferenceExample(prompt='Say hello', chosen='Hello', rejected='Farewell')


In [5]:
# Validate basic pairwise preference structure.
for example in preferences:
    assert example.prompt
    assert example.chosen
    assert example.rejected
    assert example.chosen != example.rejected

# The same response ranking can reverse under another prompt.
assert preferences[0].chosen == preferences[1].rejected
assert preferences[0].rejected == preferences[1].chosen
assert preferences[0].prompt != preferences[1].prompt

print("Prompt-conditioned preference test passed.")


Prompt-conditioned preference test passed.



### Preference Data — Takeaways

Supervised fine-tuning learns from selected reference responses.

Preference learning uses relative judgments between candidate
responses conditioned on a shared prompt.

A preference pair does not directly provide an absolute reward,
nor does it specify the probability of either response.

Response ranking may reverse when the prompt changes.

The next step is to calculate token-level and complete-response
log-probabilities under a language model.



## 02. Token vs Sequence Log-Probabilities

### Why complete-response probabilities?

A causal language model predicts one token at a time.

For a prompt x and completion y, the probability chain rule gives:

$$
\log \pi_\theta(y \mid x)
=
\sum_{t=0}^{T_y-1}
\log \pi_\theta(y_t \mid x,y_{<t}).
$$

Sequence log-probability is a sum, not a mean.

The following pipeline converts GPT logits into a score for a
complete candidate response.

```mermaid
flowchart TD
    A["Prompt x + Completion y"] --> B["Tokenize full sequence"]
    B --> C["Shift inputs and labels once"]
    C --> D["GPT forward: logits (B, T, V)"]
    D --> E["log_softmax over vocabulary V"]
    E --> F["Gather log p of actual target token"]
    F --> G["Mask prompt and padding positions"]
    G --> H["Sum completion-token log probabilities"]
    H --> I["log π(y | x)"]
```

This is teacher-forced scoring. The model receives the actual
previous response tokens when scoring each target character.

The resulting sequence log-probability is a language-model
likelihood, not a learned preference reward.


In [6]:
import torch

# (T=2,): conditional probabilities of two successive target tokens
token_probabilities = torch.tensor([0.8, 0.9])

# (): product of the conditional token probabilities
sequence_probability = token_probabilities.prod()

# (): sum of the conditional token log-probabilities
sequence_log_probability = token_probabilities.log().sum()

torch.testing.assert_close(sequence_log_probability, sequence_probability.log())

print("Sequence probability:", sequence_probability.item())
print("Sequence log-probability:", sequence_log_probability.item())


Sequence probability: 0.7199999690055847
Sequence log-probability: -0.3285040855407715


In [7]:
from pathlib import Path

from llmfp.data import (
    CharacterTokenizer,
    pad_supervised_batch,
    shift_supervised_tokens,
)
from llmfp.models import GPT, GPTConfig
from llmfp.training import per_example_sft_nll

checkpoint_path = Path("../checkpoints/tiny_shakespeare_sft/step_00040.pt")

if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Missing {checkpoint_path}. "
        "Complete the checkpoint-saving cell in Lesson 22 first."
    )

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

tokenizer = CharacterTokenizer(checkpoint["tokenizer_vocabulary"])

config = GPTConfig(**checkpoint["model_config"])

sft_policy = GPT(config)

sft_policy.load_state_dict(checkpoint["model_state_dict"], strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sft_policy = sft_policy.to(device)
sft_policy.eval()

assert tokenizer.vocab_size == config.vocab_size == 65

print("Loaded SFT checkpoint:", checkpoint_path.name)
print("Vocabulary size:", config.vocab_size)
print("Device:", device)


Loaded SFT checkpoint: step_00040.pt
Vocabulary size: 65
Device: cuda


In [8]:
candidate_examples = []
candidate_metadata = []

for pair_index, example in enumerate(preferences):
    prompt_text = f"User: {example.prompt}\nAssistant: "

    for kind, response in (
        ("chosen", example.chosen),
        ("rejected", example.rejected),
    ):
        completion_text = response + "\n"
        full_text = prompt_text + completion_text

        # (L,): serialized prompt and completion character IDs
        full_ids = torch.tensor(tokenizer.encode(full_text), dtype=torch.long)

        # (L,): full-sequence positions selected for supervision
        target_mask = torch.zeros_like(full_ids, dtype=torch.bool)

        prompt_length = len(tokenizer.encode(prompt_text))
        target_mask[prompt_length:] = True

        # (T=L-1,): causally aligned inputs and completion-only labels
        input_ids, labels = shift_supervised_tokens(full_ids, target_mask)

        # (R,): supervised response character IDs, including newline
        selected_labels = labels[labels.ne(-100)]

        torch.testing.assert_close(
            selected_labels,
            torch.tensor(tokenizer.encode(completion_text), dtype=torch.long),
        )

        candidate_examples.append((input_ids, labels))
        candidate_metadata.append(
            {
                "pair_index": pair_index,
                "kind": kind,
                "prompt": example.prompt,
                "response": response,
                "completion": completion_text,
            }
        )

# Use a valid in-vocabulary character for right padding.
pad_token_id = tokenizer.token_to_id[" "]

# (B=6, T_max): padded predictor inputs
batch_inputs, batch_labels = pad_supervised_batch(
    candidate_examples, pad_token_id=pad_token_id
)

assert batch_inputs.shape == batch_labels.shape
assert batch_inputs.shape[0] == 6
assert batch_labels.ne(-100).sum().item() == 29

print("Candidate batch:", batch_inputs.shape)
print("Total supervised characters:", 29)


Candidate batch: torch.Size([6, 35])
Total supervised characters: 29


In [9]:
import torch.nn.functional as F

# (B=6, T_max): predictor inputs on the model device
inputs_device = batch_inputs.to(device)

# (B=6, T_max): supervised labels on the model device
labels_device = batch_labels.to(device)

with torch.no_grad():
    # (B=6, T_max, V=65): GPT next-token predictions
    logits = sft_policy(inputs_device)

    # (B=6, T_max, V=65): normalized vocabulary log-probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # (B=6, T_max): supervised completion positions
    completion_mask = labels_device.ne(-100)

    # (B=6, T_max): safe indices for gathering vocabulary values
    safe_labels = labels_device.masked_fill(~completion_mask, 0)

    # (B=6, T_max): log-probability of the actual next token
    selected_log_probs = log_probs.gather(
        dim=-1, index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    # (B=6, T_max): zero out prompt and padding contributions
    token_log_probs = selected_log_probs.masked_fill(~completion_mask, 0.0)

    # (B=6,): log-probability of each full completion
    sequence_log_probs = token_log_probs.sum(dim=1)

    # (B=6,): supervised token counts per completion
    completion_lengths = completion_mask.sum(dim=1)

    # (B=6,): mean NLL, using our Lesson 22 reusable primitive
    mean_nll = per_example_sft_nll(logits, labels_device)

# The sequence score must equal minus mean NLL times token count.
torch.testing.assert_close(sequence_log_probs, -mean_nll * completion_lengths)

assert torch.isfinite(sequence_log_probs).all()

print("Sequence log-probabilities:")
for metadata, score in zip(
    candidate_metadata,
    sequence_log_probs.cpu().tolist(),
):
    print(
        f"{metadata['prompt']:12s} "
        f"{metadata['kind']:8s} "
        f"{metadata['response']:10s} "
        f"logp={score:.4f}"
    )

print("Sequence log-probability consistency passed.")


Sequence log-probabilities:
Say yes      chosen   Yes        logp=-0.0007
Say yes      rejected No         logp=-20.4530
Say no       chosen   No         logp=-18.6250
Say no       rejected Yes        logp=-0.0015
Say hello    chosen   Hello      logp=-0.0011
Say hello    rejected Farewell   logp=-10.3109
Sequence log-probability consistency passed.


In [10]:
import plotly.graph_objects as go

fig = go.Figure()

for row, kind in ((0, "chosen"), (1, "rejected")):
    metadata = candidate_metadata[row]

    # (R,): log-probabilities of actual completion characters
    values = token_log_probs[row, completion_mask[row]].cpu().tolist()

    characters = list(metadata["completion"])

    assert len(values) == len(characters)

    fig.add_trace(
        go.Bar(
            x=list(range(1, len(values) + 1)),
            y=values,
            name=f"{kind.title()}: {metadata['response']}",
            customdata=[repr(char) for char in characters],
            hovertemplate=(
                "Character position: %{x}"
                "<br>Character: %{customdata}"
                "<br>Log-probability: %{y:.4f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Token Log-Probabilities — Prompt: Say yes",
    xaxis_title="Completion character position",
    yaxis_title="Log-probability of target character",
    xaxis=dict(dtick=1),
    barmode="group",
    height=420,
)

fig.show()



### Token and Sequence Log-Probabilities — Takeaways

The probability of a completion is the product of its conditional
next-token probabilities.

Taking the logarithm converts that product into a sum.

We restored the SFT GPT from Lesson 22 and teacher-forced both
chosen and rejected responses through the same policy.

`F.log_softmax` converted vocabulary logits into log-probabilities.

`gather` selected the log-probability of each actual target token.
Ignored labels were replaced with valid gather indices and then
masked out of the result.

Summing the completion-token log-probabilities produced one
sequence log-probability per candidate response.

We verified that sequence log-probability equals negative mean
NLL multiplied by the number of supervised completion tokens.

A policy's sequence log-probability is not a preference reward.
The model is not guaranteed to prefer the annotated chosen response.

The next section examines completion masks, sequence lengths,
and the difference between sum and mean normalization.



## 03. Completion Masks and Length Effects

### Three regions in a padded preference batch

Every prediction position belongs to exactly one region:

1. Prompt/context targets: valid positions excluded from scoring.
2. Completion targets: response tokens included in scoring.
3. Right-padding targets: artificial positions excluded from scoring.

Masks are aligned with next-token prediction positions,
not the original unshifted sequence positions.

For a completion y of R tokens:

$$
S_{\mathrm{sum}}(x,y)
=
\sum_{t=1}^{R}
\log \pi_\theta(y_t \mid x,y_{<t})
$$

$$
S_{\mathrm{mean}}(x,y)
=
\frac{S_{\mathrm{sum}}(x,y)}{R}
$$

The sum is the autoregressive log-probability of the specified
completion. The mean is a length-normalized diagnostic.

In this chapter, the final newline is included in the completion.
It is an ordinary character, not a dedicated EOS token.


In [11]:
# (B=6,): original shifted sequence lengths before right padding
sequence_lengths = torch.tensor(
    [inputs.numel() for inputs, _ in candidate_examples],
    dtype=torch.long,
    device=device,
)

# (1, T_max=35): predictor position indices
position_ids = torch.arange(batch_labels.shape[1], device=device).unsqueeze(0)

# (B=6, T_max=35): original, non-padding predictor positions
valid_target_mask = position_ids < sequence_lengths.unsqueeze(1)

# (B=6, T_max=35): supervised completion prediction positions
completion_target_mask = labels_device.ne(-100)

# (B=6, T_max=35): valid predictor positions ignored as prompt/context
prompt_target_mask = valid_target_mask & ~completion_target_mask

# (B=6, T_max=35): positions introduced by right padding
padding_target_mask = ~valid_target_mask

# (B=6, T_max=35): count membership in each of three regions
region_membership = (
    prompt_target_mask.long()
    + completion_target_mask.long()
    + padding_target_mask.long()
)

assert torch.all(region_membership == 1)
assert torch.equal(completion_target_mask, completion_mask)
assert torch.all(labels_device[padding_target_mask] == -100)
assert completion_target_mask.sum().item() == 29

print("Prompt positions:", prompt_target_mask.sum().item())
print("Completion positions:", completion_target_mask.sum().item())
print("Padding positions:", padding_target_mask.sum().item())


Prompt positions: 146
Completion positions: 29
Padding positions: 35


In [ ]:
import plotly.graph_objects as go

# (B=6, T_max=35): 0=prompt, 1=completion, 2=padding
region_codes = torch.where(
    padding_target_mask, 2, torch.where(completion_target_mask, 1, 0)
)

region_names = {
    0: "Prompt/context — ignored",
    1: "Completion — scored",
    2: "Right padding — ignored",
}

region_text = [
    [region_names[value] for value in row]
    for row in region_codes.cpu().tolist()
]

candidate_labels = [
    f"{meta['prompt']} | {meta['kind']}: {meta['response']}"
    for meta in candidate_metadata
]

mask_fig = go.Figure(
    data=go.Heatmap(
        z=region_codes.cpu().tolist(),
        x=list(range(batch_labels.shape[1])),
        y=candidate_labels,
        text=region_text,
        hovertemplate=(
            "Candidate: %{y}"
            "<br>Prediction position: %{x}"
            "<br>Region: %{text}"
            "<extra></extra>"
        ),
        colorscale=[
            [0.00, "#94A3B8"],
            [0.2499, "#94A3B8"],
            [0.25, "#2563EB"],
            [0.7499, "#2563EB"],
            [0.75, "#E2E8F0"],
            [1.00, "#E2E8F0"],
        ],
        zmin=0,
        zmax=2,
        showscale=False,
        xgap=1,
        ygap=3,
    )
)

mask_fig.update_layout(
    title="Preference Batch — Prompt, Completion, and Padding",
    xaxis_title="Shifted prediction position",
    yaxis=dict(autorange="reversed"),
    height=420,
)

mask_fig.show()


In [ ]:
# (R_chosen=4,): conditional probabilities of chosen tokens
toy_chosen_probs = torch.full((4,), 0.90, dtype=torch.float64)

# (R_rejected=2,): conditional probabilities of rejected tokens
toy_rejected_probs = torch.full((2,), 0.85, dtype=torch.float64)

# (): sum of token log-probabilities for the chosen sequence
toy_chosen_sum = toy_chosen_probs.log().sum()

# (): sum of token log-probabilities for the rejected sequence
toy_rejected_sum = toy_rejected_probs.log().sum()

# (): mean token log-probabilities
toy_chosen_mean = toy_chosen_probs.log().mean()
toy_rejected_mean = toy_rejected_probs.log().mean()

print("Chosen sum:", toy_chosen_sum.item())
print("Rejected sum:", toy_rejected_sum.item())
print("Chosen mean:", toy_chosen_mean.item())
print("Rejected mean:", toy_rejected_mean.item())

# The shorter rejected completion has greater sequence probability.
assert toy_rejected_sum > toy_chosen_sum

# The chosen completion has better average per-token probability.
assert toy_chosen_mean > toy_rejected_mean

print("Length-dependent ranking reversal demonstrated.")


Chosen sum: -0.4214420626313051
Rejected sum: -0.3250378589955499
Chosen mean: -0.10536051565782628
Rejected mean: -0.16251892949777494
Length-dependent ranking reversal demonstrated.


In [14]:
# (B=6,): average log-probability per supervised completion token
average_log_probs = sequence_log_probs / completion_lengths

# Mean log-probability must equal negative per-example mean NLL.
torch.testing.assert_close(average_log_probs, -mean_nll)

for meta, length, total, average in zip(
    candidate_metadata,
    completion_lengths.cpu().tolist(),
    sequence_log_probs.cpu().tolist(),
    average_log_probs.cpu().tolist(),
):
    print(
        f"{meta['prompt']:10s} | "
        f"{meta['kind']:8s} | "
        f"length={length:2d} | "
        f"sum={total:9.4f} | "
        f"mean={average:9.4f}"
    )


Say yes    | chosen   | length= 4 | sum=  -0.0007 | mean=  -0.0002
Say yes    | rejected | length= 3 | sum= -20.4530 | mean=  -6.8177
Say no     | chosen   | length= 3 | sum= -18.6250 | mean=  -6.2083
Say no     | rejected | length= 4 | sum=  -0.0015 | mean=  -0.0004
Say hello  | chosen   | length= 6 | sum=  -0.0011 | mean=  -0.0002
Say hello  | rejected | length= 9 | sum= -10.3109 | mean=  -1.1457


In [15]:
from plotly.subplots import make_subplots

score_labels = [
    f"{meta['prompt']} | {meta['kind']}: {meta['response']}"
    for meta in candidate_metadata
]

length_values = completion_lengths.cpu().tolist()

score_fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=(
        "Complete-response log P (sum)",
        "Average log P per character",
    ),
    horizontal_spacing=0.08,
)

for column, values in (
    (1, sequence_log_probs.cpu().tolist()),
    (2, average_log_probs.cpu().tolist()),
):
    score_fig.add_trace(
        go.Bar(
            x=values,
            y=score_labels,
            orientation="h",
            customdata=length_values,
            hovertemplate=(
                "Candidate: %{y}"
                "<br>Score: %{x:.4f}"
                "<br>Completion length: %{customdata}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

score_fig.update_xaxes(title_text="Sum of log-probabilities", row=1, col=1)

score_fig.update_xaxes(title_text="Mean log-probability", row=1, col=2)

score_fig.update_yaxes(autorange="reversed", row=1, col=1)

score_fig.update_layout(
    title="Preference Candidates — Two Ways to Aggregate Token Scores",
    height=480,
    width=1100,
    showlegend=False,
)

score_fig.show()


In [16]:
# (B=6, T_max=35, V=65): copied logits for a controlled perturbation
perturbed_logits = logits.detach().clone()

# Modify only ignored predictor positions, across all vocabulary classes.
perturbed_logits[~completion_target_mask] = 50.0

# (B=6, T_max=35, V=65): normalized perturbed log-probabilities
perturbed_log_probs = F.log_softmax(perturbed_logits, dim=-1)

# (B=6, T_max=35): selected next-token log-probabilities
perturbed_selected = perturbed_log_probs.gather(
    dim=-1, index=safe_labels.unsqueeze(-1)
).squeeze(-1)

# (B=6, T_max=35): only completion positions contribute
perturbed_selected = perturbed_selected.masked_fill(
    ~completion_target_mask, 0.0
)

# (B=6,): sequence scores after ignored-position perturbation
perturbed_sequence_scores = perturbed_selected.sum(dim=1)

torch.testing.assert_close(perturbed_sequence_scores, sequence_log_probs)

print("Ignored-position logit invariance passed.")


Ignored-position logit invariance passed.



### Completion Masks and Length Effects — Takeaways

A padded preference batch contains three disjoint target regions:
prompt/context, supervised completion, and right padding.

Only completion targets contribute to sequence log-probability.

Right-padding positions and ignored prompt targets must not
contribute to response scoring.

Sequence log-probability is the sum of completion-token
log-probabilities. Mean log-probability divides this sum by
the number of supervised completion tokens.

Sum and mean normalization can rank candidate completions
differently when their lengths differ.

We demonstrated such a ranking reversal with a controlled
probability example and compared both aggregations using
our fine-tuned GPT.

Length-normalized log-probability is a diagnostic, not the
autoregressive log-probability of the complete response.
It should not silently replace sequence log-probability
inside a later preference-optimization objective.

Policy likelihood is also distinct from an explicitly learned
preference reward.

The next section derives the Bradley-Terry preference model.
